In [3]:
import json
import pandas as pd
import re
from collections import Counter

# ==================== 1. LOAD KAMUS ====================
print("📚 Loading aspect dictionary...")
with open('../data/usefuldict/final_aspects.json', 'r', encoding='utf-8') as f:
    kamus_aspek = json.load(f)

print(f"✅ Loaded {len(kamus_aspek)} aspects")
for aspect, keywords in kamus_aspek.items():
    print(f"  - {aspect}: {len(keywords)} keywords")

# ==================== 2. LOAD KOMENTAR ====================
print("\n📝 Loading comments...")
df = pd.read_csv('../data/processed/pocof7cleanedytcomment.csv')

# Cek kolom yang ada
print(f"📋 Columns available: {list(df.columns)}")

# Pilih kolom yang berisi komentar
if 'comment' in df.columns:
    comment_col = 'comment'
elif 'cleaned_comment' in df.columns:
    comment_col = 'cleaned_comment'
elif 'text' in df.columns:
    comment_col = 'text'
else:
    comment_col = df.columns[0]

print(f"📝 Using column: '{comment_col}'")

# Ambil komentar
komentar_list = df[comment_col].fillna('').astype(str).tolist()
print(f"📊 Total comments: {len(komentar_list)}")

# ==================== 3. BUAT FUNGSI DETEKSI ====================
def deteksi_aspek(teks, kamus_aspek):
    """
    Cari aspek-aspek yang disebut dalam teks
    """
    teks = teks.lower()
    
    aspek_terdeteksi = []
    keywords_terdeteksi = []  # Juga simpan keyword yang match
    
    for aspek, keywords in kamus_aspek.items():
        for keyword in keywords:
            pattern = r'\b' + re.escape(keyword.lower()) + r'\b'
            if re.search(pattern, teks):
                aspek_terdeteksi.append(aspek)
                keywords_terdeteksi.append(keyword)
                break  # Sudah ketemu aspek ini
    
    return list(set(aspek_terdeteksi)), list(set(keywords_terdeteksi))

# ==================== 4. PROSES SEMUA KOMENTAR ====================
print("\n🔍 Processing comments...")

hasil_data = []
for i, komentar in enumerate(komentar_list):  # Proses SEMUA data, bukan cuma 100
    if i % 500 == 0 and i > 0:
        print(f"  Processed {i}/{len(komentar_list)} comments...")
    
    aspek_terdeteksi, keywords_terdeteksi = deteksi_aspek(komentar, kamus_aspek)
    
    if aspek_terdeteksi:  # Hanya simpan yang ada aspeknya
        hasil_data.append({
            "comment_id": i + 1,
            "original_comment": komentar,
            "detected_aspects": ", ".join(aspek_terdeteksi),  # Gabung jadi string
            "aspect_count": len(aspek_terdeteksi),
            "matched_keywords": ", ".join(keywords_terdeteksi)  # Simpan keywords juga
        })

print(f"✅ Done! Found {len(hasil_data)} comments with aspects")
print(f"📊 Coverage: {len(hasil_data)/len(komentar_list)*100:.1f}% of comments have aspects")

# ==================== 5. KONVERSI KE DATAFRAME & SIMPAN CSV ====================
print("\n💾 Converting to DataFrame and saving CSV...")

# Buat DataFrame
hasil_df = pd.DataFrame(hasil_data)

# Urutkan kolom
column_order = ['comment_id', 'original_comment', 'detected_aspects', 
                'aspect_count', 'matched_keywords']
hasil_df = hasil_df[column_order]

# Simpan ke CSV
output_csv_path = '../data/comment_and_aspects/aspect_detection_results.csv'
hasil_df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')
print(f"📁 CSV saved to: {output_csv_path}")
print(f"📊 CSV shape: {hasil_df.shape}")

# ==================== 6. SIMPAN JUGA SEBAGAI JSON (OPTIONAL) ====================
# Kalau mau tetap punya JSON untuk referensi
output_json_path = '../data/comment_and_aspects/json/aspect_detection_results.json'
with open(output_json_path, 'w', encoding='utf-8') as f:
    # Simpan dalam format yang bisa dibaca BERT nanti
    json_results = hasil_df.to_dict(orient='records')
    json.dump(json_results, f, ensure_ascii=False, indent=2)
print(f"📁 JSON saved to: {output_json_path}")

# ==================== 7. TAMPILKAN CONTOH ====================
print("\n🔍 Sample results (5 pertama):")
print("=" * 80)
print(hasil_df.head().to_string(index=False))
print("=" * 80)

# ==================== 8. STATISTIK DETAIL ====================
print("\n📊 DETAILED STATISTICS:")
print("-" * 40)

# 1. Hitung frekuensi aspek
all_aspects = []
for aspects_str in hasil_df['detected_aspects']:
    aspects = [a.strip() for a in aspects_str.split(',')]
    all_aspects.extend(aspects)

aspek_counter = Counter(all_aspects)

print("\n1. Aspect Frequency (Top 15):")
print("-" * 30)
for aspek, count in aspek_counter.most_common(15):
    percentage = (count / len(hasil_df)) * 100
    print(f"  {aspek:20}: {count:5d} comments ({percentage:5.1f}%)")

# 2. Distribution of aspect count per comment
print("\n2. Aspects per Comment Distribution:")
print("-" * 30)
aspect_count_dist = hasil_df['aspect_count'].value_counts().sort_index()
for count, freq in aspect_count_dist.items():
    percentage = (freq / len(hasil_df)) * 100
    print(f"  {count} aspect(s): {freq:5d} comments ({percentage:5.1f}%)")

# 3. Most common aspect combinations
print("\n3. Top Aspect Combinations:")
print("-" * 30)
# Hitung kombinasi aspek
aspect_combinations = Counter()
for aspects_str in hasil_df['detected_aspects']:
    aspects = tuple(sorted([a.strip() for a in aspects_str.split(',')]))
    if len(aspects) > 1:  # Hanya kombinasi (2+ aspek)
        aspect_combinations[aspects] += 1

for combo, count in aspect_combinations.most_common(10):
    print(f"  {', '.join(combo)}: {count} comments")

print(f"\n📈 SUMMARY:")
print(f"  Total comments: {len(komentar_list)}")
print(f"  Comments with aspects: {len(hasil_df)} ({len(hasil_df)/len(komentar_list)*100:.1f}%)")
print(f"  Total aspect mentions: {len(all_aspects)}")
print(f"  Average aspects per comment: {len(all_aspects)/len(hasil_df):.2f}")

# ==================== 9. SIMPAN STATISTIK KE FILE ====================
print("\n📁 Saving statistics to file...")

stats = {
    "total_comments": int(len(komentar_list)),
    "comments_with_aspects": int(len(hasil_df)),
    "coverage_percentage": float(len(hasil_df) / len(komentar_list) * 100),
    "total_aspect_mentions": int(len(all_aspects)),
    "avg_aspects_per_comment": float(len(all_aspects) / len(hasil_df)) if len(hasil_df) > 0 else 0.0,
    "aspect_frequency": {k: int(v) for k, v in aspek_counter.most_common()},
    "aspect_count_distribution": {
        int(k): int(v) for k, v in dict(aspect_count_dist).items()
    }
}

stats_path = '../data/usefuldict/aspect_detection_stats.json'
with open(stats_path, 'w', encoding='utf-8') as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)

print(f"📁 Statistics saved to: {stats_path}")

print("\n✅ ASPECT DETECTION COMPLETE!")
print("➡️  Next step: Use 'aspect_detection_results.csv' for BERT sentiment analysis")

📚 Loading aspect dictionary...
✅ Loaded 14 aspects
  - Battery: 6 keywords
  - Camera: 6 keywords
  - Display: 43 keywords
  - Performance: 44 keywords
  - Price: 7 keywords
  - Design: 44 keywords
  - Software: 40 keywords
  - Audio: 40 keywords
  - Network: 30 keywords
  - Connectivity: 37 keywords
  - Storage: 27 keywords
  - Security: 12 keywords
  - BuildQuality: 21 keywords
  - Features: 29 keywords

📝 Loading comments...
📋 Columns available: ['clean_comment']
📝 Using column: 'clean_comment'
📊 Total comments: 7108

🔍 Processing comments...


  Processed 500/7108 comments...
  Processed 1000/7108 comments...
  Processed 1500/7108 comments...
  Processed 2000/7108 comments...
  Processed 2500/7108 comments...
  Processed 3000/7108 comments...
  Processed 3500/7108 comments...
  Processed 4000/7108 comments...
  Processed 4500/7108 comments...
  Processed 5000/7108 comments...
  Processed 5500/7108 comments...
  Processed 6000/7108 comments...
  Processed 6500/7108 comments...
  Processed 7000/7108 comments...
✅ Done! Found 2298 comments with aspects
📊 Coverage: 32.3% of comments have aspects

💾 Converting to DataFrame and saving CSV...
📁 CSV saved to: ../data/comment_and_aspects/aspect_detection_results.csv
📊 CSV shape: (2298, 5)
📁 JSON saved to: ../data/comment_and_aspects/json/aspect_detection_results.json

🔍 Sample results (5 pertama):
 comment_id                                                                                                                original_comment     detected_aspects  aspect_count     matched_ke

In [4]:
hasil_df.head()

,comment_id,original_comment,detected_aspects,aspect_count,matched_keywords
0,1,om coba minecraft java pake shders derrative p...,Display,1,minecraft
1,2,tambahin honor of kings dong om sebagai salah ...,"Display, Performance",2,"testnya, performanya"
2,4,cuma yang gua gak suka dari poco ini yaitu blo...,Security,1,blotware
3,8,thn depan auto naik harga dan ghoib,Price,1,harga
4,11,untuk jangka panjang apakah poco f ini worth it,Price,1,worth
